In [ ]:
import pandas as pd
import re 

In [ ]:
def extraer_tabla_principal(ruta_excel, hoja=0):
    # 1. Leer archivo sin encabezado fijo
    df_raw = pd.read_excel(ruta_excel, sheet_name=hoja, header=None)

    # 2. Encontrar el índice de la fila del encabezado
    header_idx = None
    for idx, row in df_raw.iterrows():
        # Busca una celda que contenga 'Folio Fiscal' o 'RFC'
        if row.astype(str).str.contains("Folio Fiscal|Folio fiscal", case=False).any():
            header_idx = idx
            break

    if header_idx is None:
        raise ValueError("No se encontró la cabecera de la tabla principal.")

    # 3. Asignar nombres de columnas y descartar filas previas
    df = df_raw.iloc[header_idx + 1 :].copy()
    df.columns = df_raw.iloc[header_idx].values

    # Limpiar nombres de columnas (espacios en blanco)
    df.columns = [
        str(col).strip() if pd.notna(col) else f"col_{i}"
        for i, col in enumerate(df.columns)
    ]

    # 4. Validar filas con patrón UUID en 'Folio Fiscal'
    # Patrón estándar de UUID: 8-4-4-4-12 caracteres hexadecimales
    uuid_pattern = re.compile(
        r"^[a-fA-F0-9]{8}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{4}-[a-fA-F0-9]{12}$"
    )

    col_folio = [c for c in df.columns if "Folio fiscal" in c or "Folio Fiscal" in c][0]

    # Filtrar solo las filas que cumplen con la estructura de un Folio Fiscal real
    df_limpio = df[
        df[col_folio]
        .astype(str)
        .str.strip()
        .apply(lambda x: bool(uuid_pattern.match(x)))
    ].copy()

    # 5. Resetear índice y convertir tipos numéricos si es necesario
    df_limpio.reset_index(drop=True, inplace=True)

    return df_limpio


# Uso
# df_final = extraer_tabla_principal('archivo_cliente.xlsx')